In [1]:
import torch.nn as nn
from girth import grm_mml
from torch.utils.data import dataset


In [2]:
import torch
from torch import nn
from torch.nn import functional as F


class VSOSpred(nn.Module):
    def __init__(self, num_usrs: int, num_olimps: int):
        super().__init__()
        self.global_strong = nn.Embedding(
            num_usrs,
            1,
        )
        self.performance = nn.Embedding(
            num_usrs * num_olimps,
            1,
        )
        nn.init.zeros_(self.global_strong.weight)
        nn.init.zeros_(self.performance.weight)
        self.num_olimps = num_olimps
    def get_strong(self, usrs_ids, olimps_ids) -> torch.Tensor:
        global_strength = self.global_strong(usrs_ids).squeeze(-1)
        joint_ids = (
            usrs_ids * self.num_olimps
            + olimps_ids
        )
        local = self.performance(
            joint_ids
        ).squeeze(-1)
        return global_strength + local
    def forward(self, usr_a_ids, usr_b_ids, olimp_ids):
        strength_a = self.get_strong(
            usr_a_ids,
            olimp_ids,
        )
        strength_b = self.get_strong(
            usr_b_ids,
            olimp_ids,
        )
        logits = strength_a - strength_b
        return {
            "logits": logits,
            "strength_a": strength_a,
            "strength_b": strength_b,
        }
    def regularization_loss(self, global_weight, offset_weight):
        global_penalty = (
            self.global_strong.weight.pow(2).mean()
        )
        offset_penalty = (
            self.performance.weight.pow(2).mean()
        )
        return (
            global_weight * global_penalty
            + offset_weight * offset_penalty
        )

In [3]:
from dataset import VSOSet
dataset = VSOSet('/home/pirojban/hahaha/LMSH_AI_HACK/src/Vlad/preprocessed_data')
dataloader = torch.utils.data.dataloader(dataset)

KeyError: 'name'

In [ ]:
num_usrs = 500

In [ ]:
model = VSOSpred(
    num_usrs=num_usrs,
    num_olimps=7,
)
optim = torch.optim.AdamW(model.parameters())
for batch in dataloader:
    out = model(
        athlete_a_ids=batch["athlete_a"],
        athlete_b_ids=batch["athlete_b"],
        competition_ids=batch["competition_id"],
    )
    pairwise_loss = F.binary_cross_entropy_with_logits(
        out["logits"],
        batch["target"].float(),
    )
    loss = (
        pairwise_loss
        + model.regularization_loss(1e-3, 1e-1)
    )
    loss.backward()
    optim.step()
    optim.zero_grad()